# Step 4. Multi-resolution binning and multi-distance stitching

Produces a **pyramid of binned datasets** (`pdata{k}_{bin}` for bin = 0 … nlevels-1) and a
corresponding reference pyramid (`pref_{bin}`).  These multi-resolution datasets allow the
iterative reconstruction in Step 6 to start at a coarse scale and progressively refine.

**Stitching algorithm (inner loop, per projection)**

Because the zone-plate optics magnify each distance differently, the four distances sample
*different physical field-of-views* of the same object.  The stitching loop works from the
most-distant (smallest magnification, largest FOV) plane down to the closest:

1. **Shift and de-magnify** each distance into the object coordinate frame using `curlySback`.
2. **Pad** each resampled image to the common object size `npsi × npsi` using smooth `linear_ramp`
   padding at the edges (keeps mean near 1 to avoid Fresnel-fringe artefacts).
3. **Intensity match** each inner distance to the previously stitched outer distance so that
   absolute intensities are consistent across the final stack.
4. **Feather-blend** overlapping regions with a quintic smooth-step weighting function,
   giving priority to the distance with higher spatial resolution (smaller index = higher mag).

> The stitched result is stored in `srdata` (shape `[ndist, npsi, npsi]`), then binned 2× per
> level and saved to `/exchange/pdata{k}_{bin}`.

In [ ]:
import numpy as np
import cupy as cp
import h5py
from holotomocupy.shift import Shift
from holotomocupy.utils import *

## Init

In [ ]:
ntheta = 1800
show = True
rotation_center_shift = -27.00227
ids = np.arange(0, 1800, 1800 / ntheta).astype('int')

path_out = '/data2/vnikitin/atomium_rec/20240924/AtomiumS2/'
file_out = f'data.h5'


with h5py.File(f'{path_out}/{file_out}') as fid:
    detector_pixelsize = fid['/exchange/detector_pixelsize'][0]    
    focusToDetectorDistance = fid['/exchange/focusdetectordistance'][0]    
    z1 = fid['/exchange/z1'][:] 
    energy = fid['/exchange/energy'][0] 
    shifts = fid['/exchange/shifts'][ids]
    attrs = fid['/exchange/attrs'][ids]
    shape = np.array(fid[f'/exchange/data0'].shape)
    shape_ref = fid['/exchange/data_white_start0'].shape
    shape_dark = fid['/exchange/data_dark0'].shape    

z2 = focusToDetectorDistance - z1
magnifications = focusToDetectorDistance / z1
norm_magnifications = magnifications / magnifications[0]
distances = (z1 * z2) / focusToDetectorDistance * norm_magnifications**2
voxelsize = detector_pixelsize / magnifications[0]
ndist = len(z1)
n = shape[1] 
npsi = int(np.ceil(n / norm_magnifications[-1] / 64)) * 64 

print(f'{energy=}')
print(f'{z1=}')
print(f'{focusToDetectorDistance=}')
print(f'{detector_pixelsize=}')
print(f'{magnifications=}')
print(f'{voxelsize=}')
print(f'{distances=}')

### Build reference pyramid

Each reference image is average-pooled 2× repeatedly to produce `nlevels` reference arrays
(`pref_0` … `pref_{nlevels-1}`).  Also read the combined shifts from Step 3 and apply the
`rotation_center_shift` correction to the horizontal component.

In [ ]:
nlevels = 4 # number of bin levels

with h5py.File(f'{path_out}/data.h5','a') as fid:       
    ref = fid[f'/exchange/pref'][:ndist]
    ref0 = ref.copy()
    for bin in range(nlevels):
        if f'/exchange/pref_{bin}' in fid:
            del fid[f'/exchange/pref_{bin}']
        fid.create_dataset(f'/exchange/pref_{bin}',data=ref0)
        ref0 = 0.5 * (ref0[..., ::2] + ref0[..., 1::2])
        ref0 = 0.5 * (ref0[..., ::2, :] + ref0[..., 1::2, :])        
            
    r = (fid[f'/exchange/cshifts_final'][ids] * n / 2048).astype('float32')

    ### compensate for the rotation center shift
    s = rotation_center_shift    
    r[...,1] += s

### Per-projection: align, stitch, and save all bin levels

Main processing loop — for each projection angle `j`:
1. Flat-field correct all distances: `rdata = data / (ref + ε)`.
2. Stitch distances from outermost to innermost (feathered blending, see header above).
3. Downsample the corrected `data[k]` by 2× for each subsequent bin level.
4. Write each bin level directly to the pre-allocated HDF5 datasets.

In [ ]:
# --- Constants precomputed once (do not depend on projection index j) ---
npad = n // 16
v = cp.linspace(0, 1, npad, endpoint=False)
v = v**5 * (126 - 420*v + 540*v**2 - 315*v**3 + 70*v**4)

# Move shifts to GPU — avoids 4500 × ndist small H2D transfers inside the loop
r_gpu = cp.array(r)

cl_shift = Shift(n, npsi, n, npsi, 1/norm_magnifications, 'complex64')
cref     = cp.array(ref)

chunk_size = 16   # projections read / written per HDF5 call

# CPU read buffers — one [chunk_size, n, n] per distance, reused each chunk
read_buf  = [np.empty([chunk_size, n, n], dtype='float32') for _ in range(ndist)]
# CPU write buffers — accumulate chunk_size projections, then flush once to HDF5
write_buf = [[np.empty([chunk_size, n // 2**b, n // 2**b], dtype='float32')
              for b in range(nlevels)] for _ in range(ndist)]

with h5py.File(f'{path_out}/data.h5', 'a') as fid:
    # Create / replace output datasets with explicit dtype
    data_out = [None] * nlevels
    for b in range(nlevels):
        for k in range(ndist):
            key = f'/exchange/pdata{k}_{b}'
            if key in fid:
                del fid[key]
        data_out[b] = [fid.create_dataset(f'/exchange/pdata{k}_{b}',
                                           shape=[ntheta, n // 2**b, n // 2**b],
                                           dtype='float32')
                       for k in range(ndist)]

    srdata = cp.zeros([ndist, npsi, npsi], dtype='float32')

    for j0 in range(0, ntheta, chunk_size):
        j1 = min(j0 + chunk_size, ntheta)
        nc = j1 - j0

        # One HDF5 call per distance fills the read buffer for the whole chunk
        for k in range(ndist):
            read_buf[k][:nc] = np.array(fid[f'/exchange/pdata{k}'][ids[j0:j1]], dtype='float32')

        for i in range(nc):
            j = j0 + i

            # Load current projection for all distances onto GPU
            data = cp.empty([ndist, n, n], dtype='float32')
            for k in range(ndist):
                data[k] = cp.array(read_buf[k][i])

            rdata = data / (cref + 1e-5)

            for k in range(ndist - 1, -1, -1):
                tmp = rdata[k].astype('complex64')
                # r_gpu[j:j+1, k] is already on GPU — no H2D transfer needed
                tmp = cl_shift.curlySback(cp.log(tmp[None]).astype('complex64'), r_gpu[j:j+1, k], k)[0].real
                tmp /= norm_magnifications[k]**2
                tmp = cp.exp(tmp)

                padx0 = int((npsi - n / norm_magnifications[k]) / 2) - int(r[j, k, 1])
                pady0 = int((npsi - n / norm_magnifications[k]) / 2) - int(r[j, k, 0])
                padx1 = int((npsi - n / norm_magnifications[k]) / 2) + int(r[j, k, 1])
                pady1 = int((npsi - n / norm_magnifications[k]) / 2) + int(r[j, k, 0])
                padx0 = min(npsi, max(0, padx0)) + 5
                pady0 = min(npsi, max(0, pady0)) + 5
                padx1 = min(npsi, max(0, padx1)) + 5
                pady1 = min(npsi, max(0, pady1)) + 5

                tmp = cp.pad(tmp[pady0:-pady1], ((pady0, pady1), (0, 0)), 'edge')
                tmp = cp.pad(tmp[:, padx0:-padx1], ((0, 0), (padx0, padx1)),
                             'linear_ramp', end_values=((1, 1), (1, 1)))

                if k < ndist - 1:
                    # Intensity match on GPU — float() does a single scalar D2H
                    mmm = float(srdata[k+1][pady0:-pady1, padx0:-padx1].mean() /
                                tmp[pady0:-pady1, padx0:-padx1].mean())
                    tmp     *= mmm
                    data[k] *= mmm

                    # v is precomputed; wx/wy are quick 1-D fills
                    wx = cp.ones(npsi, dtype='float32')
                    wy = cp.ones(npsi, dtype='float32')
                    wx[:padx0] = 0;  wx[padx0:padx0+npad] = v;  wx[-padx1-npad:-padx1] = 1-v;  wx[-padx1:] = 0
                    wy[:pady0] = 0;  wy[pady0:pady0+npad] = v;  wy[-pady1-npad:-pady1] = 1-v;  wy[-pady1:] = 0

                    w   = cp.outer(wy, wx)
                    tmp = tmp * w + srdata[k+1] * (1 - w)
                srdata[k] = tmp

            if j % 100 == 0:
                st, en = npsi // 2 - npsi // 8, npsi // 2 + npsi // 8
                mmean = srdata[:, st:en, st:en].mean(axis=(1, 2)) / srdata[0, st:en, st:en].mean()
                print(mmean)
                mshow_complex(srdata[0] + 1j*srdata[ndist-1], show, vmax=1.5, vmin=0.5)

            # Downsample and accumulate into write buffer (GPU→CPU once per bin×distance)
            for k in range(ndist):
                datak = data[k]
                for b in range(nlevels):
                    write_buf[k][b][i] = datak.get()
                    datak = 0.5*(datak[::2, :] + datak[1::2, :])
                    datak = 0.5*(datak[:, ::2] + datak[:, 1::2])

        # Flush chunk — one write per bin×distance instead of chunk_size writes
        for k in range(ndist):
            for b in range(nlevels):
                data_out[b][k][j0:j1] = write_buf[k][b][:nc]